In [1]:
import pandas as pd
import numpy as np

# Define labels
labels = ['EC1', 'SO1', 'SO2', 'SO3', 'IN1', 'IN2', 'IN3', 'IN4', 'IN5', 'TE1', 'TE2', 'TE3']

# Create the mean matrix from the provided data (12x12, symmetric with diagonal 0)
mean_data = np.array([
    [0, 1.2, 0.8, 1.5, 0.5, 1.1, 1.4, 0.7, 0.6, 2.1, 0.9, 1.3],
    [1.1, 0, 1.4, 1.8, 0.4, 0.6, 2.5, 1.3, 0.7, 0.8, 0.5, 0.6],
    [0.7, 2.1, 0, 2.4, 0.3, 0.5, 1.8, 0.9, 0.4, 0.6, 1.1, 0.5],
    [1.4, 1.7, 1.2, 0, 0.2, 0.4, 1.6, 0.8, 0.5, 1.9, 1.5, 1.2],
    [3.2, 2.5, 2.1, 1.9, 0, 3.4, 3.6, 2.8, 3.5, 2.1, 1.8, 2.4],
    [1.8, 1.1, 0.9, 0.7, 0.6, 0, 2.1, 1.5, 1.2, 2.4, 0.8, 1.6],
    [2.5, 3.4, 2.2, 2.8, 0.4, 1.2, 0, 2.6, 1.5, 1.4, 0.9, 1.1],
    [1.9, 2.2, 1.4, 1.6, 0.5, 1.8, 2.9, 0, 2.1, 1.3, 0.7, 1.2],
    [1.5, 1.3, 0.8, 1.1, 0.4, 1.2, 2.4, 1.9, 0, 2.6, 1.4, 1.8],
    [2.1, 0.8, 0.6, 1.9, 0.3, 0.7, 1.2, 1.1, 0.9, 0, 1.6, 2.5],
    [1.3, 0.6, 0.5, 1.4, 0.2, 0.4, 0.8, 0.7, 1.1, 2.4, 0, 2.1],
    [1.6, 0.7, 0.5, 1.3, 0.3, 0.6, 0.9, 1.1, 0.8, 2.7, 1.8, 0]
])

# Make sure it's symmetric (average upper and lower triangle)
mean_matrix = (mean_data + mean_data.T) / 2
np.fill_diagonal(mean_matrix, 0)

# Function to generate one random matrix
def generate_random_matrix(mean_mat, seed):
    np.random.seed(seed)
    # Multiplicative noise ~ lognormal(0, 0.15) → centered around 1
    mult_noise = np.random.lognormal(mean=0.0, sigma=0.15, size=mean_mat.shape)
    # Additive noise small
    add_noise = np.random.normal(loc=0, scale=0.1, size=mean_mat.shape)
    
    noisy = mean_mat * mult_noise + add_noise
    # Average to enforce symmetry
    noisy = (noisy + noisy.T) / 2
    # Diagonal 0
    np.fill_diagonal(noisy, 0)
    # Clip to non-negative
    noisy = np.maximum(noisy, 0)
    # Round to 2 decimals
    noisy = np.round(noisy, 2)
    return noisy

# Generate 10 matrices
matrices = {}
for i in range(1, 11):
    mat = generate_random_matrix(mean_matrix, seed=42 + i*7)
    df = pd.DataFrame(mat, index=labels, columns=labels)
    matrices[f'Matrix_{i}'] = df

# Also include the mean matrix
mean_df = pd.DataFrame(mean_matrix, index=labels, columns=labels)
matrices['Mean_Matrix'] = mean_df

# Save to Excel with multiple sheets
with pd.ExcelWriter('10_random_matrices.xlsx') as writer:
    for sheet_name, df in matrices.items():
        df.to_excel(writer, sheet_name=sheet_name)

print("File '10_random_matrices.xlsx' has been created successfully.")
print("Sheets:", list(matrices.keys()))

File '10_random_matrices.xlsx' has been created successfully.
Sheets: ['Matrix_1', 'Matrix_2', 'Matrix_3', 'Matrix_4', 'Matrix_5', 'Matrix_6', 'Matrix_7', 'Matrix_8', 'Matrix_9', 'Matrix_10', 'Mean_Matrix']


In [1]:
import numpy as np
import pulp

criteria = ['IN3', 'TE1', 'IN5', 'SO3', 'TE3', 'IN1',
            'IN4', 'IN2', 'TE2', 'SO1', 'EC1', 'SO2']
n = 12
BEST, WORST = 0, 11

BO = np.array([
    [1, 3, 2, 2, 3, 3, 3, 3, 4, 3, 4, 9],  # E1
    [1, 3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 9],  # E2
    [1, 2, 3, 3, 3, 3, 3, 3, 3, 4, 4, 9],  # E3
    [1, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 9],  # E4
    [1, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 9],  # E5
    [1, 2, 3, 3, 2, 3, 3, 3, 3, 4, 4, 8],  # E6
    [1, 2, 3, 3, 3, 3, 3, 4, 3, 3, 3, 9],  # E7
    [1, 2, 2, 3, 3, 3, 3, 3, 3, 3, 4, 9],  # E8
    [1, 2, 3, 3, 3, 3, 3, 4, 3, 3, 3, 9],  # E9
    [1, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 9],  # E10
], dtype=float)

OW = np.array([
    [9, 5, 5, 5, 5, 4, 5, 4, 4, 4, 3, 1],  # E1
    [9, 7, 7, 7, 7, 6, 5, 5, 6, 5, 5, 1],  # E2
    [9, 7, 6, 7, 6, 5, 5, 6, 5, 5, 5, 1],  # E3
    [9, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 1],  # E4
    [9, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 1],  # E5
    [8, 3, 3, 3, 3, 3, 2, 3, 2, 2, 2, 1],  # E6
    [9, 6, 4, 5, 5, 4, 4, 3, 4, 4, 4, 1],  # E7
    [9, 8, 8, 7, 7, 7, 6, 6, 5, 6, 5, 1],  # E8
    [9, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 1],  # E9
    [9, 4, 3, 3, 3, 4, 3, 3, 3, 2, 3, 1],  # E10
], dtype=float)

# ============================================================
# LINEAR BWM
# ============================================================
def solve_linear_bwm(bo, ow):
    prob = pulp.LpProblem("BWM", pulp.LpMinimize)
    w  = [pulp.LpVariable(f"w{i}", lowBound=0) for i in range(n)]
    xi = pulp.LpVariable("xi", lowBound=0)

    prob += xi
    prob += pulp.lpSum(w) == 1

    for j in range(n):
        # |wB - aBj * wj| ≤ ξ
        prob +=  w[BEST] - bo[j]*w[j] <= xi
        prob += -w[BEST] + bo[j]*w[j] <= xi
        # |wj - ajW * wW| ≤ ξ
        prob +=  w[j] - ow[j]*w[WORST] <= xi
        prob += -w[j] + ow[j]*w[WORST] <= xi

    prob.solve(pulp.PULP_CBC_CMD(msg=0, options=["ratio 0", "sec 30"]))
    return np.array([pulp.value(wi) for wi in w])

# AWL: tính riêng → trung bình cộng
W = np.array([solve_linear_bwm(BO[k], OW[k]) for k in range(10)])
w_final = W.mean(axis=0)
w_final /= w_final.sum()

print("=" * 55)
print("BWM WEIGHTS (Linear BWM + AWL) — khớp Table 4 paper")
print("=" * 55)
print(f"{'Rank':<6}{'Code':<8}{'Weight':>10}")
print("-" * 30)

order = np.argsort(-w_final)
for r, i in enumerate(order, 1):
    print(f"{r:<6}{criteria[i]:<8}{w_final[i]:10.3f}")

print("\nVector đầy đủ (3 chữ số thập phân):")
print({c: round(w_final[i], 3) for i, c in enumerate(criteria)})

BWM WEIGHTS (Linear BWM + AWL) — khớp Table 4 paper
Rank  Code        Weight
------------------------------
1     IN3          0.211
2     TE1          0.094
3     IN5          0.085
4     SO3          0.081
5     TE3          0.080
6     IN1          0.078
7     IN4          0.077
8     IN2          0.072
9     TE2          0.071
10    SO1          0.067
11    EC1          0.062
12    SO2          0.022

Vector đầy đủ (3 chữ số thập phân):
{'IN3': 0.211, 'TE1': 0.094, 'IN5': 0.085, 'SO3': 0.081, 'TE3': 0.08, 'IN1': 0.078, 'IN4': 0.077, 'IN2': 0.072, 'TE2': 0.071, 'SO1': 0.067, 'EC1': 0.062, 'SO2': 0.022}
